# Command Pattern
## Design Patterns for the AI Era · Part 6

> **Bad Code → Better Design → Working Python → AI Application**

Encapsulate agent actions as objects. Enable audit, retry, queuing, and undo.

## 1. The Problem

An AI agent calls tools. The naïve approach is a giant if/elif dispatcher with no history, no audit, no undo.

In [ ]:
# ❌ BAD CODE

def execute_action_bad(action: str, params: dict):
    """Problems: no history, no undo, no retry, no audit, must edit to add tools."""
    if action == 'search':
        return f'search results for {params.get("query")}'
    elif action == 'send_email':
        print(f'[EMAIL] Sending to {params.get("to")}')
        return {'sent': True}
    elif action == 'create_record':
        print(f'[DB] Creating {params.get("type")}')
        return {'id': 'REC-001'}
    else:
        raise ValueError(f'Unknown action: {action}')

execute_action_bad('search', {'query': 'design patterns'})
execute_action_bad('send_email', {'to': 'team@co.com', 'subject': 'Hi'})

## 2. What's Wrong?

- No record of what happened — no audit trail
- Cannot retry a failed action
- Cannot undo anything
- Cannot queue actions for later
- Cannot add authorization checks without touching the dispatch function
- Adding a tool means editing the same function again

In [ ]:
# ✅ GOOD CODE — Command Pattern
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any

@dataclass
class CommandResult:
    success: bool
    output: Any = None
    error: str = ''


class Command(ABC):
    @abstractmethod
    def execute(self) -> CommandResult: ...

    def can_undo(self) -> bool:
        return False

    def undo(self) -> CommandResult:
        raise NotImplementedError(f'{self.__class__.__name__} does not support undo')


@dataclass
class SearchCommand(Command):
    query: str
    top_k: int = 5

    def execute(self) -> CommandResult:
        results = [f'doc about {self.query} #{i}' for i in range(self.top_k)]
        return CommandResult(success=True, output=results)


@dataclass
class SendEmailCommand(Command):
    to: str
    subject: str
    body: str = ''

    def execute(self) -> CommandResult:
        print(f'[EMAIL] Sending to {self.to}: {self.subject}')
        return CommandResult(success=True, output={'sent_to': self.to})


@dataclass
class CreateRecordCommand(Command):
    record_type: str
    data: dict
    _created_id: str = field(default='', init=False, repr=False)

    def execute(self) -> CommandResult:
        self._created_id = f'REC-{hash(str(self.data)) % 1000:03d}'
        print(f'[DB] Created {self.record_type}: {self._created_id}')
        return CommandResult(success=True, output={'id': self._created_id})

    def can_undo(self) -> bool:
        return bool(self._created_id)

    def undo(self) -> CommandResult:
        print(f'[DB] Deleting {self.record_type}: {self._created_id}')
        self._created_id = ''
        return CommandResult(success=True, output={'deleted': True})


# Executor with history
class AgentExecutor:
    def __init__(self):
        self._history: list[tuple[Command, CommandResult]] = []

    def run(self, command: Command) -> CommandResult:
        result = command.execute()
        self._history.append((command, result))
        return result

    def undo_last(self) -> CommandResult | None:
        for cmd, _ in reversed(self._history):
            if cmd.can_undo():
                result = cmd.undo()
                return result
        return None

    def history(self):
        return [(type(cmd).__name__, res.success) for cmd, res in self._history]


executor = AgentExecutor()
executor.run(SearchCommand(query='factory pattern', top_k=3))
executor.run(SendEmailCommand(to='team@co.com', subject='Summary'))
executor.run(CreateRecordCommand(record_type='Case', data={'subject': 'Bug'}))

print('\nHistory:', executor.history())
print('\nUndo last undoable command:')
executor.undo_last()

## 4. Tests

In [ ]:
# ✅ Tests

def test_search_command():
    cmd = SearchCommand(query='python', top_k=3)
    result = cmd.execute()
    assert result.success
    assert len(result.output) == 3
    print('PASS test_search_command')

def test_executor_records_history():
    ex = AgentExecutor()
    ex.run(SearchCommand(query='test', top_k=1))
    ex.run(SearchCommand(query='test2', top_k=1))
    assert len(ex.history()) == 2
    print('PASS test_executor_records_history')

def test_create_record_undo():
    ex = AgentExecutor()
    ex.run(CreateRecordCommand(record_type='Account', data={'name': 'ACME'}))
    undo_result = ex.undo_last()
    assert undo_result is not None
    assert undo_result.success
    print('PASS test_create_record_undo')

def test_search_command_cannot_undo():
    cmd = SearchCommand(query='x')
    assert not cmd.can_undo()
    print('PASS test_search_command_cannot_undo')

def test_command_result_on_failure():
    class FailCommand(Command):
        def execute(self):
            try:
                raise RuntimeError('boom')
            except Exception as e:
                return CommandResult(success=False, error=str(e))

    ex = AgentExecutor()
    result = ex.run(FailCommand())
    assert not result.success
    assert 'boom' in result.error
    print('PASS test_command_result_on_failure')

test_search_command()
test_executor_records_history()
test_create_record_undo()
test_search_command_cannot_undo()
test_command_result_on_failure()
print('\nAll tests passed ✅')

## 5. AI Example — Dispatching LLM Tool Calls (Claude-style)

In [ ]:
# 🤖 AI EXAMPLE — Map Claude tool_use blocks to Command objects

COMMAND_REGISTRY = {
    'search':        lambda p: SearchCommand(**{k: p[k] for k in p if k in ('query', 'top_k')}),
    'send_email':    lambda p: SendEmailCommand(**{k: p[k] for k in p if k in ('to', 'subject', 'body')}),
    'create_record': lambda p: CreateRecordCommand(record_type=p['type'], data=p.get('data', {})),
}

def dispatch_tool_call(tool_name: str, tool_input: dict, executor: AgentExecutor) -> CommandResult:
    factory = COMMAND_REGISTRY.get(tool_name)
    if factory is None:
        return CommandResult(success=False, error=f'Unknown tool: {tool_name}')
    command = factory(tool_input)
    return executor.run(command)


# Simulate tool_use blocks from a Claude response
tool_calls = [
    {'name': 'search',     'input': {'query': 'design patterns', 'top_k': 3}},
    {'name': 'send_email', 'input': {'to': 'lead@co.com', 'subject': 'Report ready'}},
    {'name': 'unknown_tool','input': {}},
]

ex = AgentExecutor()
for call in tool_calls:
    result = dispatch_tool_call(call['name'], call['input'], ex)
    status = '✅' if result.success else '❌'
    print(f'{status} {call["name"]}: {result.output or result.error}')

print('\nFull history:', ex.history())

## 6. Summary

| | Before | After |
|---|---|---|
| Adding a tool | Edit dispatch function | New Command class + register |
| Audit trail | None | `executor.history()` |
| Undo support | None | `cmd.undo()` per-command |
| Testing | All tools entangled | Each Command in isolation |
| Authorization | Add to dispatch function | Pre-execute hook on executor |